In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 00 — Environment Setup and Holdout Isolation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import drive\n",
    "drive.mount('/content/drive')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import shutil\n",
    "from pathlib import Path\n",
    "\n",
    "REPO_URL = \"https://github.com/EnomisLP/DiverseVul--IS-Project\"\n",
    "REPO_DIR = \"/content/vuln-detection\"\n",
    "\n",
    "if os.path.isdir(REPO_DIR):\n",
    "    !git clone {REPO_URL} {REPO_DIR}\n",
    "else:\n",
    "    !git -C {REPO_DIR} pull origin main\n",
    "\n",
    "print(f\"Cloning repository directly into {REPO_DIR}...\")\n",
    "!git clone {REPO_URL} {REPO_DIR}\n",
    "\n",
    "%cd {REPO_DIR}\n",
    "\n",
    "!pip install -q -e ."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import shutil\n",
    "from pathlib import Path\n",
    "\n",
    "\n",
    "DRIVE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/')\n",
    "RAW_DATA_TARGET_DIR = Path('data/raw')\n",
    "RAW_DATA_TARGET_DIR.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "if DRIVE_DIR.exists():\n",
    "    candidates = list(DRIVE_DIR.glob('*.jsonl')) + list(DRIVE_DIR.glob('*.json')) + list(DRIVE_DIR.glob('*.csv'))\n",
    "    if candidates:\n",
    "        source_file = candidates[0]\n",
    "        target_file_path = RAW_DATA_TARGET_DIR / source_file.name\n",
    "        if not target_file_path.exists():\n",
    "            shutil.copy(source_file, target_file_path)\n",
    "            print(f'Staged dataset: {target_file_path}')\n",
    "        else:\n",
    "            print(f'Dataset already exists at destination: {target_file_path}')\n",
    "    else:\n",
    "        print('No valid dataset file found in Colab Notebooks folder.')\n",
    "else:\n",
    "    print('Drive not mounted correctly or folder path is wrong.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "import pandas as pd\n",
    "from pathlib import Path\n",
    "\n",
    "source_path = Path('/content/vuln-detection/data/raw/RDiverseVul.json')\n",
    "target_path = Path('/content/vuln-detection/data/raw/RDiverseVul.jsonl')\n",
    "\n",
    "if source_path.exists():\n",
    "    print(f\"Reading column-oriented JSON from {source_path.name}...\")\n",
    "    \n",
    "    \n",
    "    with open(source_path, 'r', encoding='utf-8') as f:\n",
    "        raw_data = json.load(f)\n",
    "        \n",
    "    \n",
    "    df = pd.DataFrame(raw_data)\n",
    "    \n",
    "    \n",
    "    if df.shape[0] == 6 and df.shape[1] > 100:\n",
    "        print(\"Detected inverted shape matrix. Transposing data...\")\n",
    "        df = df.T\n",
    "        \n",
    "    print(f\"Success! Corrected dataframe shape: {df.shape}\")\n",
    "    print(f\"Verified Columns: {df.columns.tolist()}\")\n",
    "    \n",
    "    print(f\"Saving cleanly to line-delimited format: {target_path.name}...\")\n",
    "    \n",
    "    df.to_json(target_path, orient='records', lines=True)\n",
    "    \n",
    "    print(\"File conversion complete! Run your data loader cell next.\")\n",
    "else:\n",
    "    print(\"Raw 'RDiverseVul.json' file not found.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "from pathlib import Path\n",
    "\n",
    "REPO_ROOT = Path('/content/vuln-detection/vuln-detection')\n",
    "if REPO_ROOT.exists():\n",
    "    os.chdir(str(REPO_ROOT))\n",
    "    if str(REPO_ROOT) not in sys.path:\n",
    "        sys.path.append(str(REPO_ROOT))\n",
    "\n",
    "from src.data import load_rdiversevul, make_holdout_split, load_split_config\n",
    "\n",
    "df = load_rdiversevul(data_dir='/content/vuln-detection/data/raw', drop_duplicates=True)\n",
    "split_params = load_split_config(config_path='configs/track_a.yaml')\n",
    "\n",
    "try:\n",
    "    train_idx, holdout_idx = make_holdout_split(\n",
    "        df=df,\n",
    "        test_size=split_params['test_size'],\n",
    "        random_state=split_params['random_state'],\n",
    "        overwrite=False\n",
    "    )\n",
    "    print(f'Successfully completed splits. Train size: {len(train_idx)} | Holdout size: {len(holdout_idx)}')\n",
    "except FileExistsError:\n",
    "    print('Split files already exist. Skipping split creation.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "assert torch.cuda.is_available(), 'CRITICAL: GPU acceleration required for Track B.'\n",
    "print(f\"GPU verified: {torch.cuda.get_device_name(0)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from transformers import AutoConfig, AutoTokenizer\n",
    "from src.utils import load_config\n",
    "\n",
    "!pip install -q xformers\n",
    "\n",
    "cfg_b = load_config('/content/vuln-detection/vuln-detection/configs/track_b.yaml')\n",
    "\n",
    "model_identifier = cfg_b.backbone.name\n",
    "\n",
    "print(f\"Fetching and caching assets for: {model_identifier}\")\n",
    "\n",
    "tokenizer = AutoTokenizer.from_pretrained(model_identifier, trust_remote_code=True)\n",
    "config = AutoConfig.from_pretrained(model_identifier, trust_remote_code=True)\n",
    "\n",
    "print(\"Successfully cached configuration and tokenizer!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import platform\n",
    "import psutil\n",
    "from src.data import load_holdout_split\n",
    "\n",
    "train_idx, holdout_idx = load_holdout_split()\n",
    "print(f'Python: {platform.python_version()}')\n",
    "print(f'PyTorch: {torch.__version__}')\n",
    "print(f'RAM: {psutil.virtual_memory().total / 1e9:.2f} GB')\n",
    "print(f'Train size: {len(train_idx)} | Holdout size: {len(holdout_idx)}')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}